# Análise do algoritmo Selection Sort

Este notebook executa o Selection Sort em Cython e registra quantas vezes as principais linhas conceituais são executadas.

A atividade pede comparar as entradas:

- `(1, 2, 3, 4, 5)`;
- `(5, 4, 3, 2, 1)`.

O objetivo é perceber que o número de comparações não muda quando a entrada já está ordenada.

In [ ]:
%load_ext cython

## Pseudocódigo analisado

```text
SELECTION-SORT(A)
1 for i = 1 to A.comprimento - 1
2     menor = i
3     for j = i+1 to A.comprimento
4         if A[j] < A[menor]
5             menor = j
6     trocar A[i] com A[menor]
```

A linha mais importante para o crescimento do custo é a comparação da linha 4, pois ela ocorre para todos os pares relevantes de `i` e `j`.

## Análise assintótica

Para cada posição `i`, o algoritmo procura o menor elemento no restante do vetor.

O número de comparações é:

\[
(n-1) + (n-2) + \cdots + 1 = 
rac{n(n-1)}{2}
\]

Portanto, o tempo é **Θ(n²)** em todos os casos: melhor, médio e pior.

O número de atribuições em `menor = j` muda conforme a entrada, mas essa diferença não altera a classe assintótica, porque o custo dominante continua sendo o laço interno.

In [ ]:
%%cython
# cython: boundscheck=False, wraparound=False, cdivision=True

def selection_sort_com_contagem(list entrada):
    cdef list A = list(entrada)
    cdef Py_ssize_t n = len(A)
    cdef Py_ssize_t i, j, menor
    cdef long linha1 = 0, linha2 = 0, linha3 = 0, linha4 = 0, linha5 = 0, linha6 = 0
    cdef object temp

    for i in range(n - 1):
        linha1 += 1
        menor = i
        linha2 += 1

        for j in range(i + 1, n):
            linha3 += 1
            linha4 += 1
            if A[j] < A[menor]:
                menor = j
                linha5 += 1

        temp = A[i]
        A[i] = A[menor]
        A[menor] = temp
        linha6 += 1

    return A, {
        "linha1_for_externo": linha1,
        "linha2_menor_i": linha2,
        "linha3_for_interno": linha3,
        "linha4_comparacao": linha4,
        "linha5_atualizacao_menor": linha5,
        "linha6_troca": linha6,
    }

In [ ]:
casos = {
    "ordenado": [1, 2, 3, 4, 5],
    "decrescente": [5, 4, 3, 2, 1],
}

for nome, vetor in casos.items():
    ordenado, contagem = selection_sort_com_contagem(vetor)
    print("
Caso:", nome)
    print("Entrada:", vetor)
    print("Saída:", ordenado)
    for linha, vezes in contagem.items():
        print(f"{linha:28s}: {vezes}")

## Conclusão da comparação

Para `n = 5`, a linha de comparação executa:

\[

rac{5(5-1)}{2} = 10
\]

Isso ocorre tanto no vetor ordenado quanto no vetor decrescente.

A entrada altera quantas vezes `menor` é atualizado, mas não altera o número de comparações feitas pelo algoritmo.

In [ ]:
for n in [5, 10, 100]:
    comparacoes = n * (n - 1) // 2
    print(f"n={n:3d} -> comparações = {comparacoes}")